# 07 — Ajuste de hiperparámetros

Ajusta los tres candidatos mejor posicionados en la CV anterior: gradient boosting, random forest y SVC RBF. Para cada uno evalúa 30 configuraciones con los mismos 3 folds estratificados sobre desarrollo. El holdout y el test de competencia no intervienen.

## MLflow

Inicia el servidor local con `bash scripts/start_mlflow.sh` desde PSet3. Cada configuración se registra como una corrida hija y cada búsqueda como una corrida padre en el experimento `pset3-churn-fine-tuning`. La métrica de selección es ROC-AUC; también se registran average precision, F1, precision, recall y balanced accuracy.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "datos.py").exists():
    raise FileNotFoundError("Abre este notebook desde la carpeta PSet3.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ajuste import run_hyperparameter_tuning
from src.datos import load_obt
from src.preparacion import split_churn_data

In [2]:
obt = load_obt()
obt.columns = obt.columns.str.lower()
split = split_churn_data(obt)
tuning_results = run_hyperparameter_tuning(split)
display(tuning_results.round(4))

2026/11/26 23:12:58 INFO mlflow.tracking.fluent: Experiment with name 'pset3-churn-fine-tuning' does not exist. Creating a new experiment.


🏃 View run gradient_boosting_config_01 at: http://localhost:5001/#/experiments/6/runs/66f4fbc921304396b659184b97372248
🧪 View experiment at: http://localhost:5001/#/experiments/6
🏃 View run gradient_boosting_config_02 at: http://localhost:5001/#/experiments/6/runs/4227400cdfb8440cba0b06719efd7c98
🧪 View experiment at: http://localhost:5001/#/experiments/6
🏃 View run gradient_boosting_config_03 at: http://localhost:5001/#/experiments/6/runs/a5a2c133d10a4acab3e3c3a50e77aa6e
🧪 View experiment at: http://localhost:5001/#/experiments/6
🏃 View run gradient_boosting_config_04 at: http://localhost:5001/#/experiments/6/runs/7a397f39bf3b4610a041ae1fab7138a3
🧪 View experiment at: http://localhost:5001/#/experiments/6
🏃 View run gradient_boosting_config_05 at: http://localhost:5001/#/experiments/6/runs/cbdb054e6d6646da993924dd5ebfdcac
🧪 View experiment at: http://localhost:5001/#/experiments/6
🏃 View run gradient_boosting_config_06 at: http://localhost:5001/#/experiments/6/runs/5395c2cf08ec4bf28f5

,model,best_params,configurations,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,balanced_accuracy_mean,balanced_accuracy_std
0,gradient_boosting,"{""model__learning_rate"": 0.07, ""model__max_dep...",30,0.9317,0.0026,0.8198,0.0039,0.7225,0.0070,0.7897,0.0075,0.6661,0.0169,0.8100,0.0069
1,random_forest,"{""model__max_depth"": 15, ""model__min_samples_l...",30,0.9238,0.0023,0.7921,0.0030,0.7227,0.0026,0.6739,0.0029,0.7792,0.0026,0.8405,0.0016
2,svc_rbf,"{""model__C"": 10.0, ""model__class_weight"": ""bal...",30,0.9214,0.0019,0.7817,0.0062,0.6849,0.0033,0.5725,0.0119,0.8530,0.0213,0.8435,0.0049


## Selección para evaluación final

Compara los mejores ROC-AUC de las tres búsquedas y revisa las métricas secundarias antes de decidir los dos modelos que pasarán al holdout. No vuelvas a usar el holdout para cambiar la búsqueda.